In [72]:
import vsdx
import json
from collections import deque
color_type_mapping = {"#fff2cc": "Question Node", "#9dbb61": "Yes Node", "#c05046": "No Node", "#b7dde8": "Termination Node", "#e5b9b5" : "Conditions to respect Node", "#ddd6e5": "IF collector node"}

def parse_with_library(filepath, start_id: str = None):
    nodes, edges = [], []

    with vsdx.VisioFile(filepath) as vis:
        for page in vis.pages:
            for s in page.all_shapes:
      
                is_connector = any(c.from_id == s.ID for c in s.connects)
                if is_connector:
                    begin = next((c for c in s.connects if c.from_rel == 'BeginX'), None)
                    end = next((c for c in s.connects if c.from_rel == 'EndX'), None)
                    edges.append({
                        "id": s.ID,
                        "label": s.text,
                        "from": begin.to_id if begin else None,
                        "to": end.to_id if end else None,
                    })
                else:
                    if s.text: 
                        override_type = None
                        mp_name = s.master_page.name if s.master_page else "Unknown"
                        if mp_name == "Subprocess":
                            print(f"mPName = {mp_name}  ID = {s.ID}  text = {s.text}  fill_color = {s.fill_color}"  )
                            if s.fill_color == None:
                                override_type = "Intermediate Conclusion"
                            elif s.fill_color == "#e5b9b5":
                                override_type = "Obligations Node"
                        elif mp_name == "Start/End":
                            if s.text.lower().replace("\n", " ").strip() == "if":
                                override_type = "IF collector node"
                   
                        nodes.append({
                            "id": s.ID,
                            "label": s.text,
                            "shape_type": s.shape_type,
                            "color": s.fill_color,
                            "mp_name": mp_name,
                            "reverse_engineering_type": override_type if override_type else color_type_mapping.get(s.fill_color, "unknown"),
                            "x": s.x,
                            "y": s.y,
                        })

    if not start_id:
        return {"nodes": nodes, "edges": edges}

    node_map = {n["id"]: n for n in nodes}

    if start_id not in node_map:
        raise ValueError(f"Node ID '{start_id}' not found. Available IDs: {list(node_map.keys())}")

    adjacency = {n["id"]: [] for n in nodes}
    for e in edges:
        if e["from"] in adjacency:
            adjacency[e["from"]].append(e)

    visited_nodes = set()
    visited_edges = set()
    queue = deque([start_id])

    while queue:
        current_id = queue.popleft()
        if current_id in visited_nodes:
            continue
        visited_nodes.add(current_id)

        for edge in adjacency.get(current_id, []):
            visited_edges.add(edge["id"])
            if edge["to"] and edge["to"] not in visited_nodes:
                queue.append(edge["to"])

    return {
        "start": start_id,
        "nodes": [n for n in nodes if n["id"] in visited_nodes],
        "edges": [e for e in edges if e["id"] in visited_edges],
    }

In [73]:
res = parse_with_library("flowchart.vsdx")
with open("flowchart.json", "w") as f:
    json.dump(res, f, indent=4)

mPName = Subprocess  ID = 303  text = Your dataset does not contain personal data
  fill_color = None
mPName = Subprocess  ID = 308  text = Your dataset contains personal data
  fill_color = None
mPName = Subprocess  ID = 605  text = Ensure that you are respecting the following obligations:
  fill_color = #e5b9b5
mPName = Subprocess  ID = 645  text = Ensure that you are respecting the following obligations:
  fill_color = #fbd7bb


In [15]:
with open("flowchart.json") as f:
    data = json.load(f)

In [ ]:
import graphviz



dot = graphviz.Digraph(graph_attr={"rankdir": "TB", "splines": "ortho"},
                       node_attr={"shape": "box", "style": "rounded,filled", "fillcolor": "#e8f4f8", "fontsize": "10"},
                       edge_attr={"fontsize": "9"})

for node in data["nodes"]:
    label = (node["label"] or "").strip() or node["id"]
    dot.node(node["id"], label=label, fillcolor=node.get("color", "#e8f4f8"), tooltip=node.get("reverse_engineering_type", "unknown"))

for edge in data["edges"]:
    if edge["from"] and edge["to"]:
        label = (edge["label"] or "").strip()
        dot.edge(edge["from"], edge["to"], label=label)

dot.render("flowchart", format="pdf", cleanup=True)

'flowchart.pdf'

In [ ]:
def parse_if_nodes(flowchart_data):
    node_map = {n["id"]: n for n in flowchart_data["nodes"]}

    outgoing = {}
    for e in flowchart_data["edges"]:
        outgoing.setdefault(e["from"], []).append(e)

    if_nodes = []
    for node in flowchart_data["nodes"]:
        if node.get("reverse_engineering_type") != "IF collector node":
            continue

        node_id = node["id"]
        if_collector_parents = [e["from"] for e in flowchart_data["edges"] if e["to"] == node_id]
        unique_start_nodes = list(set(
            e["from"] for e in flowchart_data["edges"] if e["to"] in if_collector_parents
        ))
        start_nodes = [n for n in unique_start_nodes if n not in if_collector_parents]

        or_pairs  = [[e["from"], e["to"]] for e in flowchart_data["edges"] if e["from"] in if_collector_parents and (e["label"] or "").lower().replace("\n", " ").strip() == "or"]
        and_pairs = [[e["from"], e["to"]] for e in flowchart_data["edges"] if e["from"] in if_collector_parents and (e["label"] or "").lower().replace("\n", " ").strip() == "and"]

        output_nodes = [e["to"] for e in outgoing.get(node_id, [])]

        if_nodes.append({
            "type": "conditions",
            "id": node_id,
            "label": node["label"],
            "conditions": if_collector_parents,
            "condition_details": [node_map[c] for c in if_collector_parents if c in node_map],
            "or_conditions": list(set(x for pair in or_pairs for x in pair if x != node_id)),
            "and_conditions": list(set(x for pair in and_pairs for x in pair if x != node_id)),
            "start_nodes": start_nodes,
            "start_node_details": [node_map[n] for n in start_nodes if n and n in node_map],
            "output_nodes": output_nodes,
            "output_node_details": [node_map[n] for n in output_nodes if n and n in node_map],
        })

    for simple in parse_simple_if_nodes(flowchart_data):
        if_nodes.append({**simple, "type": "simple"})

    return if_nodes


In [ ]:
def parse_simple_if_nodes(flowchart_data):
    node_map = {n["id"]: n for n in flowchart_data["nodes"]}

    yes_node_ids = {n["id"] for n in flowchart_data["nodes"] if n.get("reverse_engineering_type") == "Yes Node"}
    no_node_ids  = {n["id"] for n in flowchart_data["nodes"] if n.get("reverse_engineering_type") == "No Node"}
    if_collector_ids = {n["id"] for n in flowchart_data["nodes"] if n.get("reverse_engineering_type") == "IF collector node"}

    outgoing = {}
    for e in flowchart_data["edges"]:
        outgoing.setdefault(e["from"], []).append(e)

    incoming = {}
    for e in flowchart_data["edges"]:
        incoming.setdefault(e["to"], []).append(e)

    simple_ifs = []
    seen_parents = set()

    for yes_id in yes_node_ids:
        for edge in incoming.get(yes_id, []):
            parent_id = edge["from"]
            if parent_id in seen_parents or parent_id in if_collector_ids:
                continue
            seen_parents.add(parent_id)

            no_id = next(
                (e["to"] for e in outgoing.get(parent_id, []) if e["to"] in no_node_ids),
                None,
            )

            yes_next = [e["to"] for e in outgoing.get(yes_id, [])]
            no_next  = [e["to"] for e in outgoing.get(no_id, [])] if no_id else []

            simple_ifs.append({
                "condition_node": parent_id,
                "condition_node_info": node_map.get(parent_id),
                "yes_node": yes_id,
                "yes_node_info": node_map.get(yes_id),
                "yes_next": yes_next,
                "yes_next_details": [node_map[n] for n in yes_next if n in node_map],
                "no_node": no_id,
                "no_node_info": node_map.get(no_id),
                "no_next": no_next,
                "no_next_details": [node_map[n] for n in no_next if n in node_map],
            })

    return simple_ifs


In [ ]:
parse_simple_if_nodes(data)

In [40]:
parse_if_nodes(data)

[{'id': '217',
  'label': 'IF\n',
  'conditions': ['184', '186', '188', '198', '204', '207', '233', '235'],
  'or_conditions': ['186',
   '235',
   '233',
   '189',
   '198',
   '207',
   '188',
   '184',
   '204'],
  'and_conditions': [],
  'start_nodes': ['18', '189']},
 {'id': '343',
  'label': 'IF\n',
  'conditions': ['332', '337'],
  'or_conditions': ['332', '337'],
  'and_conditions': [],
  'start_nodes': ['329']},
 {'id': '392',
  'label': 'IF\n',
  'conditions': ['374', '375', '379', '377', '376', '370'],
  'or_conditions': [],
  'and_conditions': [],
  'start_nodes': ['365']},
 {'id': '424',
  'label': 'IF\n',
  'conditions': ['416', '417'],
  'or_conditions': ['417', '416'],
  'and_conditions': [],
  'start_nodes': ['414']},
 {'id': '510',
  'label': 'IF\n',
  'conditions': ['492', '493', '494', '495'],
  'or_conditions': ['494', '493', '495', '492'],
  'and_conditions': [],
  'start_nodes': ['488']},
 {'id': '584',
  'label': 'IF\n',
  'conditions': ['30', '38', '44', '49', 

In [69]:
import json

class FlowchartNavigator:
    def __init__(self, flowchart_data, start_id="1"):
        self.data = flowchart_data
        self.node_map = {n["id"]: n for n in flowchart_data["nodes"]}

        self.outgoing = {}
        self.incoming = {}
        for e in flowchart_data["edges"]:
            self.outgoing.setdefault(e["from"], []).append(e)
            self.incoming.setdefault(e["to"],   []).append(e)

        if_nodes = parse_if_nodes(flowchart_data)
        self.collector_if_map = {n["id"]: n             for n in if_nodes if n["type"] == "conditions"}
        self.simple_if_map    = {n["condition_node"]: n  for n in if_nodes if n["type"] == "simple"}

        self._start_id = str(start_id)

    def export_html(self, path="flowchart_navigator.html"):
        nav_data = {
            "startId":  self._start_id,
            "nodeMap":  self.node_map,
            "outgoing": {nid: [{"to": e["to"]} for e in edges]
                         for nid, edges in self.outgoing.items()},
            "collectorIfMap": {
                nid: {k: ctx[k] for k in
                      ("conditions","condition_details","or_conditions","and_conditions","output_nodes")}
                for nid, ctx in self.collector_if_map.items()
            },
            "simpleIfMap": {
                nid: {"yes_node": ctx["yes_node"], "no_node": ctx["no_node"]}
                for nid, ctx in self.simple_if_map.items()
            },
        }

        html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>GDPR Flowchart Navigator</title>
<style>
*, *::before, *::after { box-sizing: border-box; }
body { font-family: sans-serif; margin: 0; padding: 24px; background: #f8f9fa; color: #222; }
h1 { font-size: 1.3em; margin: 0 0 20px; color: #333; }
#cards { display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 16px; }
.card {
    border: 2px solid rgba(0,0,0,.15); border-radius: 10px; padding: 16px;
    min-width: 240px; max-width: 480px;
    box-shadow: 0 1px 4px rgba(0,0,0,.08);
}
.card-title { font-size: 1em; font-weight: bold; margin-bottom: 3px; }
.card-meta  { font-size: .75em; color: #444; margin-bottom: 12px; }
.clist { display: flex; flex-direction: column; gap: 5px; margin-bottom: 10px; }
.citem {
    display: block; width: 100%; padding: 9px 13px;
    border: 1px solid #ccc; border-radius: 6px; background: #f5f5f5;
    cursor: pointer; font-size: .87em; text-align: left; font-family: inherit;
    transition: background .12s, border-color .12s;
}
.citem:hover  { background: #e8e8e8; }
.citem.active { background: #d4edda; border-color: #5cb85c; font-weight: 600; }
.status       { font-size: .84em; margin: 7px 0; }
.status.met   { color: #1a7a3a; font-weight: bold; }
.status.unmet { color: #888; }
.terminal     { color: #1a7a3a; font-weight: bold; margin-top: 8px; font-size: .9em; }
.yesno        { display: flex; gap: 8px; margin-top: 10px; }
.btn {
    padding: 7px 18px; border-radius: 6px; border: none; cursor: pointer;
    font-size: .9em; font-family: inherit; font-weight: 500;
    transition: opacity .12s, transform .08s;
}
.btn:disabled              { opacity: .35; cursor: default; }
.btn:not([disabled]):hover  { opacity: .85; }
.btn:not([disabled]):active { transform: scale(.97); }
.btn-yes     { background: #28a745; color: #fff; }
.btn-no      { background: #dc3545; color: #fff; }
.btn-proceed { background: #28a745; color: #fff; margin-top: 10px; }
.btn-proceed[disabled] { background: #aaa; }
.btn-next    { background: #007bff; color: #fff; }
.btn-restart { background: #ffc107; color: #333; }
#controls    { display: flex; gap: 10px; align-items: center; }
.done        { color: #1a7a3a; font-weight: bold; }
</style>
</head>
<body>
<h1>GDPR Flowchart Navigator</h1>
<div id="cards"></div>
<div id="controls"></div>
<script>
const DATA = """ + json.dumps(nav_data) + """;

let state = { activeIds: [DATA.startId], checks: {} };

const isSimple    = n => n in DATA.simpleIfMap;
const isCollector = n => n in DATA.collectorIfMap;
const isDecision  = n => isSimple(n) || isCollector(n);
const isTerminal  = n => { const o = DATA.outgoing[n]; return !o || !o.length; };

function evalConds(id) {
    const ctx = DATA.collectorIfMap[id], ch = state.checks[id] || {};
    const orC = ctx.or_conditions || [], andC = ctx.and_conditions || [];
    if (!orC.length && !andC.length) return (ctx.conditions || []).some(c => ch[c]);
    return (!orC.length || orC.some(c => ch[c])) && (!andC.length || andC.every(c => ch[c]));
}

function stepAll() {
    const seen = new Set(), next = [];
    for (const nid of state.activeIds) {
        if (isDecision(nid) || isTerminal(nid)) {
            if (!seen.has(nid)) { next.push(nid); seen.add(nid); } continue;
        }
        const ns = (DATA.outgoing[nid] || []).map(e => e.to).filter(Boolean);
        for (const n of (ns.length ? ns : [nid])) if (!seen.has(n)) { next.push(n); seen.add(n); }
    }
    state.activeIds = next; render();
}

function answer(nodeId, choice) {
    const ctx = DATA.simpleIfMap[nodeId];
    const target = choice === 'yes' ? ctx.yes_node : ctx.no_node;
    const seen = new Set(), next = [];
    for (const nid of state.activeIds) {
        const dest = nid === nodeId ? target : nid;
        if (!seen.has(dest)) { next.push(dest); seen.add(dest); }
    }
    state.activeIds = next; render();
}

function resolveConditions(ifId) {
    const outputs = DATA.collectorIfMap[ifId].output_nodes || [];
    const seen = new Set(), next = [];
    for (const nid of state.activeIds)
        for (const t of (nid === ifId ? outputs : [nid]))
            if (!seen.has(t)) { next.push(t); seen.add(t); }
    state.activeIds = next; render();
}

function toggleCondition(ifId, condId) {
    if (!state.checks[ifId]) state.checks[ifId] = {};
    state.checks[ifId][condId] = !state.checks[ifId][condId];
    render();
}

function reset() { state = { activeIds: [DATA.startId], checks: {} }; render(); }

document.addEventListener('click', e => {
    const btn = e.target.closest('[data-action]');
    if (!btn || btn.hasAttribute('disabled')) return;
    const d = btn.dataset;
    if      (d.action === 'toggleCondition')   toggleCondition(d.ifid, d.condid);
    else if (d.action === 'answer')             answer(d.nodeid, d.choice);
    else if (d.action === 'resolveConditions') resolveConditions(d.ifid);
    else if (d.action === 'stepAll')            stepAll();
    else if (d.action === 'reset')              reset();
});

function esc(s) {
    return String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;').replace(/"/g,'&quot;');
}

function buildCard(nid) {
    const node = DATA.nodeMap[nid]; if (!node) return '';
    const label = esc((node.label || '').replace(/\n/g,' ').trim()) || '(no label)';
    const ntype = esc(node.reverse_engineering_type || 'unknown');
    const color = node.color || '#f0f0f0';
    let body = '';

    if (isCollector(nid)) {
        const ctx = DATA.collectorIfMap[nid];
        const orS  = new Set(ctx.or_conditions  || []);
        const andS = new Set(ctx.and_conditions || []);
        const ch   = state.checks[nid] || {};
        let items  = '';
        for (const cd of (ctx.condition_details || [])) {
            const cl  = esc((cd.label || '').replace(/\n/g,' ').trim());
            const tag = andS.has(cd.id) ? ' <b>[AND]</b>' : orS.has(cd.id) ? ' <b>[OR]</b>' : '';
            items += `<button class="citem${ch[cd.id] ? ' active' : ''}"
                data-action="toggleCondition" data-ifid="${nid}" data-condid="${cd.id}">${cl}${tag}</button>`;
        }
        const sat = evalConds(nid);
        body = `<div class="clist">${items}</div>
            <div class="status ${sat ? 'met' : 'unmet'}">${sat ? '&#10003; Conditions met' : '&#9675; Not yet met'}</div>
            <button class="btn btn-proceed"${sat ? '' : ' disabled'} data-action="resolveConditions" data-ifid="${nid}">Proceed &#8594;</button>`;

    } else if (isSimple(nid)) {
        body = `<div class="yesno">
            <button class="btn btn-yes" data-action="answer" data-nodeid="${nid}" data-choice="yes">Yes</button>
            <button class="btn btn-no"  data-action="answer" data-nodeid="${nid}" data-choice="no">No</button>
            </div>`;

    } else if (isTerminal(nid)) {
        body = '<div class="terminal">&#10003; Terminal</div>';
    }

    return `<div class="card" style="background:${color}">
        <div class="card-title">${label}</div>
        <div class="card-meta">${ntype} &middot; ID: ${nid}</div>
        ${body}</div>`;
}

function render() {
    const canAdv = state.activeIds.some(n => !isDecision(n) && !isTerminal(n));
    const done   = state.activeIds.length > 0 && state.activeIds.every(isTerminal);
    document.getElementById('cards').innerHTML =
        state.activeIds.map(buildCard).join('');
    document.getElementById('controls').innerHTML =
        (canAdv ? '<button class="btn btn-next" data-action="stepAll">Next &#8594;</button>' : '')
        + (done  ? '<span class="done">&#10003; All paths complete.</span>' : '')
        + '<button class="btn btn-restart" data-action="reset">&#8635; Restart</button>';
}

render();
</script>
</body>
</html>"""

        with open(path, "w", encoding="utf-8") as f:
            f.write(html)
        print(f"Saved: {path}")

    def __repr__(self):
        return f"<FlowchartNavigator start={self._start_id}>"


In [70]:
nav = FlowchartNavigator(data, start_id="1")
nav.export_html("flowchart_navigator.html")

Saved: flowchart_navigator.html
